# Comparison between normal traffic and IBR traffic

This Jupyter notebook compares the characteristics of normal traffic and IBR traffic.

The script uses the conversation output from `Wireshark` to generate a histogram of the number of packets per conversation. The histogram is then compared between normal traffic.

First:

Extract the conversations using the menu in `Wireshark`: `Statistics` -> `Conversations` -> Select the protocol (TCP, UDP, etc.) -> Click on `Copy` button to copy the conversations to clipboard -> Paste the conversations into a text file.

If files are too big, use `tshark` to extract the conversations and the script `plot_conversation_histogram_tshark.ipynb` to plot the histogram.

As normal traffic, we are using  `Monday-WorkingHours-stats-tcp.txt` and `Monday-WorkingHours-stats-tcp.txt` files.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# TCP input files
normal_traffic_tcp = 'Monday-WorkingHours-stats-tcp.txt'
normal_traffic_udp = 'Monday-WorkingHours-stats-udp.txt'

# UDP input files
ibr_traffic_tcp = 'darknet_20251010_tcp_conversations.csv'
ibr_traffic_udp = 'darknet_20251010_udp_conversations.csv'

# PDF output files
output_pdf_tcp = 'comparison_flows_generic_ibr_tcp.pdf'
output_pdf_udp = 'comparison_flows_generic_ibr_udp.pdf'

In [ ]:
# These key names are used in the Wireshark conversation statistics
key_1 = 'Packets A → B'
key_2 = 'Packets B → A'

In [ ]:
# Function to process and draw a chart from a CSV file
def plot_packet_histogram(csv_file, ax, title):
    # Read data
    df = pd.read_csv(csv_file, sep=',')

    # Define intervals
    bins = [0, 1, 5, 20, 100, float('inf')]
    labels = ['1', '2-5', '6-20', '21-100', '>100']

    # Classify each value into its interval for both columns
    df['bin_A_B'] = pd.cut(df[key_1], bins=bins, labels=labels, right=True, include_lowest=True)
    df['bin_B_A'] = pd.cut(df[key_2], bins=bins, labels=labels, right=True, include_lowest=True)

    # Calculate totals per bin (summing packets)
    packets_A_B = df.groupby('bin_A_B', observed=False)[key_1].sum()
    packets_B_A = df.groupby('bin_B_A', observed=False)[key_2].sum()

    # Create DataFrame and calculate percentages
    hist_df = pd.DataFrame({'Received': packets_A_B, 'Replied': packets_B_A}).fillna(0).reindex(labels)
    total_packets = hist_df['Received'].sum() + hist_df['Replied'].sum()
    hist_df_percent = hist_df / total_packets * 100

    # Draw stacked chart on the provided axis
    hist_df_percent.plot(
        kind='bar',
        stacked=True,
        ax=ax,
        width=0.85,
        color=['#1f77b4', '#ff7f0e'],
        legend=False  # shared legend outside the function
    )

    # Add labels above bars
    for idx, (entrada, salida) in enumerate(zip(hist_df_percent['Received'], hist_df_percent['Replied'])):
        total = entrada + salida
        label = f"({entrada:.1f}%/{salida:.1f}%)"
        ax.text(idx, total + 1, label, ha='center', va='bottom', fontsize=9, fontweight='bold', rotation=90)

    # Styling
    ax.set_ylim(0, 130)
    ax.set_title(title)
    # ax.set_xlabel('Packet in the flow (Received/Replied)')
    ax.set_xlabel('')
    ax.set_ylabel('Total packets (%)', fontsize=16)

    # Rotation of xticks in x-axis
    ax.set_xticklabels(labels, rotation=90, fontsize=16)

In [ ]:
def generate_graphic(file1, title1, file2, title2, output_file):
    # Create figure with two subplots in a row
    fig, axes = plt.subplots(1, 2, figsize=(6, 5), sharey=True)

    # Draw both histograms
    plot_packet_histogram(file1, axes[0], title1)
    plot_packet_histogram(file2, axes[1], title2)

    # Shared legend
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2, title='')

    plt.tight_layout(rect=[0, 0, 1, 0.95])  # leave space for legend above

    # Save the figure in the file comparison_flows_generic_ibr_tcp.pdf
    fig.savefig(output_file)

    plt.show()

In [ ]:
generate_graphic(normal_traffic_tcp, '(a) Generic traffic', ibr_traffic_tcp, '(b) IBR traffic', output_pdf_tcp)

In [ ]:
generate_graphic(normal_traffic_udp, '(a) Generic traffic', ibr_traffic_udp, '(b) IBR traffic', output_pdf_udp)